In [ ]:
# Xem toàn bộ fields của qa_pairs

print("All fields in qa_pairs[0]:")

for k, v in qa_pairs[0].items():

    print(f"  {k}: {repr(v)[:100]}")
 
print("\n---")

print("All fields in qa_pairs[1]:")

for k, v in qa_pairs[1].items():

    print(f"  {k}: {repr(v)[:100]}")
 
# Xem corpus có bao nhiêu pages per domain

from collections import Counter

domains = [item['domain'] for item in qa_pairs]

print(f"\nQueries per domain:")

print(Counter(domains))
 
# Xem GT indices range per domain — để biết mỗi domain có bao nhiêu pages

from collections import defaultdict

domain_gt_max = defaultdict(int)

domain_gt_min = defaultdict(lambda: 99999)

for item in qa_pairs:

    gt = item.get('gt_relevance', item['gt_embed_indices'])

    gt_indices = list(gt.keys()) if isinstance(gt, dict) else list(gt)

    d = item['domain']

    domain_gt_max[d] = max(domain_gt_max[d], max(gt_indices))

    domain_gt_min[d] = min(domain_gt_min[d], min(gt_indices))
 
print(f"\nGT index range per domain:")

for d in sorted(domain_gt_max):

    print(f"  {d}: [{domain_gt_min[d]}, {domain_gt_max[d]}]")
 
# ==============================================================================

# SETUP: Build domain -> page index mapping (chạy 1 lần trước METHOD 1)

# ==============================================================================

print("Building domain -> page index mapping...")
 
from collections import defaultdict
 
# all_page_embeddings[i] tương ứng với page index i trong corpus

# qa_pairs có field 'domain' và 'gt_embed_indices' (global indices)

# Cần biết mỗi global index thuộc domain nào
 
# Lấy domain của từng page dựa vào qa_pairs

# GT indices của queries trong 1 domain đều thuộc domain đó

page_to_domain = {}

for item in qa_pairs:

    domain = item['domain']

    gt = item.get('gt_relevance', item['gt_embed_indices'])

    gt_indices = list(gt.keys()) if isinstance(gt, dict) else list(gt)

    for idx in gt_indices:

        page_to_domain[idx] = domain
 
# Verify coverage

print(f"Pages mapped to domain: {len(page_to_domain)} / {len(all_page_embeddings)}")
 
# Build domain -> sorted list of global page indices

domain_to_pages = defaultdict(list)

for page_idx, domain in page_to_domain.items():

    domain_to_pages[domain].append(page_idx)
 
for d, pages in sorted(domain_to_pages.items()):

    pages.sort()

    print(f"  {d}: {len(pages)} pages  [{min(pages)} .. {max(pages)}]")
 
# Pre-build per-domain doc matrix (một lần, tái dùng cho tất cả queries)

print("\nPre-building per-domain doc matrices...")

domain_doc_matrix  = {}   # domain -> (doc_matrix, doc_mask)

domain_local_index = {}   # domain -> list of global page indices (sorted)
 
for domain, global_indices in domain_to_pages.items():

    global_indices_sorted = sorted(global_indices)

    domain_local_index[domain] = global_indices_sorted
 
    domain_embs = [all_page_embeddings[i] for i in global_indices_sorted]

    d_mat, d_mask = build_doc_matrix(domain_embs, device)

    domain_doc_matrix[domain] = (d_mat, d_mask)

    print(f"  {domain}: matrix {d_mat.shape}")
 
print("✅ Domain matrices ready")
 
 
# ==============================================================================

# METHOD 1 — Traditional MaxSim  (domain-filtered, matches ViDoRe v3 leaderboard)

# ==============================================================================

print("\n>>> METHOD 1: Traditional MaxSim  [domain-filtered]")
 
trad_metrics        = {}

trad_domain_metrics = {}

trad_query_rows     = []

trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD    = ['traditional']
 
pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")
 
for q_idx, item in pbar:

    question = item['question']

    gt_set   = item.get('gt_relevance', item['gt_embed_indices'])

    domain   = item['domain']
 
    # ── Encode query live ────────────────────────────────────────────────────

    q_inputs = query_processor.process_queries([question]).to(device)

    with torch.no_grad():

        q_proj = query_model(**q_inputs)           # (1, S, dim)

    attn_mask = q_inputs['attention_mask'][0]      # (S,)

    trad_idx  = torch.where(attn_mask > 0)[0]

    q_emb     = q_proj[0][trad_idx].float()        # (N_active, dim)

    q_norm    = F.normalize(q_emb, dim=-1)
 
    # ── Domain-filtered retrieval ────────────────────────────────────────────

    doc_matrix, doc_mask   = domain_doc_matrix[domain]   # pre-built matrix cho domain này

    global_indices_sorted  = domain_local_index[domain]  # global page idx tương ứng

    n_domain_docs          = doc_matrix.shape[0]
 
    torch.cuda.synchronize()

    t_score_start = time.perf_counter()
 
    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)           # (N_active, n_domain_docs)

    scores = M.sum(dim=0)                                         # (n_domain_docs,)

    top10_local = torch.topk(scores, min(10, n_domain_docs)).indices.cpu().tolist()
 
    torch.cuda.synchronize()

    score_ms = (time.perf_counter() - t_score_start) * 1000.0

    trad_latency.add_ratio(1.0, score_ms)
 
    # ── Convert local indices -> global page indices ─────────────────────────

    top10_global = [global_indices_sorted[i] for i in top10_local]
 
    # ── Metrics ──────────────────────────────────────────────────────────────

    # gt_set dùng global indices, top10_global cũng là global -> hit_metrics đúng

    m = hit_metrics(top10_global, gt_set)

    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)
 
    trad_query_rows.append({

        'query_id':       q_idx,

        'doc_name':       item['doc_name'],

        'domain':         domain,

        'question':       question,

        'trad_r@1':       m['r1'],

        'trad_r@5':       m['r5'],

        'trad_r@10':      m['r10'],

        'trad_ndcg@1':    round(m['n1'],  4),

        'trad_ndcg@5':    round(m['n5'],  4),

        'trad_ndcg@10':   round(m['n10'], 4),

    })
 
print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,

              title="Traditional MaxSim Results  [domain-filtered]")

trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(

    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)

print("\n✅ Saved: traditional_queries.csv")
 